# 10k-iteration paper grid -- **Account 3 of 4**

Self-contained. **Runtime > Change runtime type > T4 GPU**, then Run All.

This account's jobs, in order:

| # | backbone | mode | protocol | seed | est. |
|---|---|---|---|---|---|
| 1 | `efficientnet_b4` | `none` | `random` | 42 | ~85 min |
| 2 | `xception` | `none` | `identity` | 43 | ~50 min |
| 3 | `xception` | `hfc_only` | `identity` | 43 | ~50 min |

Total ~185 min (~3.1 h). Free Colab will probably reclaim the session before
that finishes -- **that is fine**. Checkpoints land on Drive every 1000
iterations, so when you get a fresh runtime just Run All again: finished jobs
are skipped and the interrupted one resumes where it stopped.

Run names carry `_s<seed>_i10k` and results go to `results10k/`, so none of
this collides with the existing 4000-iteration numbers.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 1. Code and dependencies

In [ ]:
REPO = "https://github.com/rahulbhatyogesh/DIP-Project.git"

import os

if os.path.exists("/content/code"):
    !cd /content/code && git pull -q
else:
    !git clone -q {REPO} /content/code
%cd /content/code

!pip install -q timm albumentations 2>&1 | tail -2
import timm, albumentations
print("timm", timm.__version__, "| albumentations", albumentations.__version__)

In [ ]:
# Architecture check -- must match the paper's Table IV exactly before we spend
# four GPU-hours on it.
from src.models.build import verify_param_counts
assert verify_param_counts(), "parameter counts do not match the paper"
print("param counts match Table IV")

## 2. Drive, for checkpoints that survive a reclaimed session

In [ ]:
# Checkpoints go to Drive so a reclaimed session loses minutes, not hours.
from google.colab import drive
drive.mount("/content/drive")

import os
RUNS = "/content/drive/MyDrive/dip-hfc-runs"
os.makedirs(RUNS, exist_ok=True)
print("checkpoints ->", RUNS)

## 3. Face crops

From the private Kaggle dataset built in `01_preprocess_kaggle.ipynb`. Needs your API token: kaggle.com > Settings > **Create New Token**. Upload `kaggle.json` when prompted.

In [ ]:
# Pin the version explicitly. Kaggle clients default to the NEWEST version, so
# the empty version 2 will 404 while the good data sits in version 1.
DATASET = "rahulbhat05/celebdf-hfc-faces/versions/1"

from pathlib import Path
import json, os

if not Path("/content/faces/manifest.csv").exists():
    # kagglehub reads credentials from ~/.kaggle/kaggle.json or the environment.
    # With none it calls the API anonymously, and a PRIVATE dataset answers that
    # with 403 Forbidden -- not a login prompt. So ask for the token first.
    #
    # NOTE: this must be the token of the Kaggle account that OWNS the dataset
    # (rahulbhat05), on every Colab account. Colab identity != Kaggle identity.
    cred = Path.home() / ".kaggle" / "kaggle.json"
    if not cred.exists():
        from google.colab import files
        print("Upload kaggle.json  (kaggle.com > Settings > Create New Token)")
        up = files.upload()
        assert up, "no file uploaded -- re-run this cell"
        cred.parent.mkdir(parents=True, exist_ok=True)
        cred.write_bytes(up[next(iter(up))])   # handles 'kaggle (1).json' too
        cred.chmod(0o600)
    tok = json.loads(cred.read_text())
    os.environ["KAGGLE_USERNAME"] = tok["username"]
    os.environ["KAGGLE_KEY"] = tok["key"]
    print("kaggle auth as:", tok["username"])

    !pip install -q kagglehub
    import kagglehub
    src = Path(kagglehub.dataset_download(DATASET))
    print("downloaded to:", src)
    faces = next(src.rglob("manifest.csv")).parent
    if not Path("/content/faces").exists():
        os.symlink(faces, "/content/faces")   # symlink, not a 2.3 GB copy

assert Path("/content/faces/manifest.csv").exists(), "crops not found"
assert Path("/content/faces/splits.json").exists(), "splits.json missing"
# -L because /content/faces is a symlink and find will not descend one otherwise.
!echo "crops: $(find -L /content/faces -name '*.jpg' | wc -l)"

## 4. Train and evaluate

One loop over this account's 3 jobs. Each trains to 10,000 iterations then scores the **test** partition.

Safe to re-run at any point -- it skips what is already done.

Early validation numbers look bad because BatchNorm running statistics need a few hundred iterations to settle. Judge the trend, not the first number.

In [ ]:
import json, subprocess, time
from pathlib import Path

RUNS, FACES, ITERS = "/content/drive/MyDrive/dip-hfc-runs", "/content/faces", 10000
RESULTS = RUNS + "/results10k"
Path(RESULTS).mkdir(parents=True, exist_ok=True)


def stream(cmd):
    """Run cmd, forwarding its output into the notebook as it arrives.

    A subprocess inherits the kernel's real fd 1, which Colab does not display
    -- only Python-level print() reaches the cell.  So pipe the child and
    re-print each line.  "-u" stops Python block-buffering its stdout when that
    stdout is a pipe rather than a terminal; without it output arrives in 8 KB
    lumps, many minutes late.

    Interrupting a Colab cell raises KeyboardInterrupt in the kernel but does
    NOT kill this child.  Left alone it keeps training, and a second run of the
    cell then puts two trainers on the same --out: interleaved log.jsonl, and
    two processes racing to write best.pt.  So kill it on the way out.
    """
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in p.stdout:
            print(line, end="", flush=True)
    except (KeyboardInterrupt, Exception):
        p.terminate()
        try:
            p.wait(timeout=30)
        except subprocess.TimeoutExpired:
            p.kill()
        raise
    return p.wait()


# (backbone, mode, protocol, seed) -- ordered so Tier 1 finishes first.
JOBS = [
    ("efficientnet_b4", "none", "random", 42),
    ("xception", "none", "identity", 43),
    ("xception", "hfc_only", "identity", 43),
]

for n, job in enumerate(JOBS, 1):
    bk, mode, proto, seed = job
    run = bk + "_" + mode + "_" + proto + "_s" + str(seed) + "_i10k"
    out, res = RUNS + "/" + run, RESULTS + "/" + run + ".json"
    print("\n" + "=" * 70)
    print("[%d/%d] %s" % (n, len(JOBS), run))
    print("=" * 70, flush=True)

    if Path(res).exists():
        print("  already complete -- skipping")
        continue

    t0 = time.time()
    if not Path(out + "/summary.json").exists():
        rc = stream([
            "python", "-u", "-m", "src.train",
            "--faces-root", FACES,
            "--manifest", FACES + "/manifest.csv",
            "--splits", FACES + "/splits.json",
            "--backbone", bk, "--mode", mode, "--protocol", proto,
            "--seed", str(seed), "--max-iters", str(ITERS),
            "--select-by", "auc", "--workers", "2",
            "--out", out, "--resume",
        ])
        if rc != 0:
            print("  !! training exited %d -- re-run this cell to resume" % rc)
            break
    else:
        print("  training already finished -- evaluating")

    stream([
        "python", "-u", "-m", "src.eval", "--ckpt", out + "/best.pt",
        "--faces-root", FACES,
        "--manifest", FACES + "/manifest.csv",
        "--splits", FACES + "/splits.json",
        "--part", "test", "--out", res,
    ])
    print("  done in %.1f min" % ((time.time() - t0) / 60))

print("\nall jobs for this account attempted")

## 5. Collect

Paste the output of this cell back into the chat.

In [ ]:
# Everything this account produced, ready to paste back into the chat.
import json
from pathlib import Path

RESULTS = "/content/drive/MyDrive/dip-hfc-runs/results10k"
print("#" * 70)
print("# ACCOUNT 3 RESULTS -- copy everything below this line")
print("#" * 70)
for p in sorted(Path(RESULTS).glob("*_i10k.json")):
    print("\n--- " + p.stem + " ---")
    print(json.dumps(json.load(open(p)), indent=2))

## 6. Curves (optional)

In [ ]:
# Validation curves: does 10k actually buy anything over 4k?
import json
from pathlib import Path
import matplotlib.pyplot as plt

RUNS = "/content/drive/MyDrive/dip-hfc-runs"
logs = sorted(Path(RUNS).glob("*_i10k/log.jsonl"))
if not logs:
    print("no 10k runs finished yet")
else:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for lp in logs:
        rec = [json.loads(l) for l in open(lp)]
        it = [r["iteration"] for r in rec]
        name = lp.parent.name.replace("_i10k", "")
        ax[0].plot(it, [r["val"]["loss"] for r in rec], label=name)
        ax[1].plot(it, [r["val"]["frame"]["auc"] for r in rec], label=name)
    ax[0].set_xlabel("iteration"); ax[0].set_ylabel("val BCE loss")
    ax[0].set_yscale("log"); ax[0].set_title("validation loss")
    ax[1].set_xlabel("iteration"); ax[1].set_ylabel("val frame AUC")
    ax[1].set_title("validation AUC")
    ax[1].legend(fontsize=7, loc="lower right")
    plt.tight_layout()
    plt.savefig(RUNS + "/curves_10k_account3.png", dpi=150)
    plt.show()